In [ ]:
import pandas as pd
import tqdm, os
import pickle
from sklearn.preprocessing import StandardScaler
import warnings
warnings.simplefilter("ignore")

import sys
sys.path.insert(0, "../..")

from utils.data_processing import dataset_df_to_token_df
from utils.acoustics_processing import get_mfccs
from utils.articulation_processing import get_aeema_articulatory_features

In [ ]:
df = pd.read_csv("../data_processing/dataset_construction/datasets/processed_aeema.csv")
df["subject"] = df["art_f"].apply(lambda x: os.path.basename(x).split("_")[2])
token_df = dataset_df_to_token_df(df)

In [5]:
token_df

,start,end,phone,art_f,ac_f,subject
0,23.743016,23.803016,ɪ,/Volumes/Transcend/USC-TIMIT/EMA/Data/F5/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F5/wav/u...,f5
1,21.513000,21.563000,n,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3
2,14.333447,14.403447,h,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
3,16.067234,16.117234,f,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
4,7.165941,7.225941,ɝ,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F1/wav/u...,f1
...,...,...,...,...,...,...
56877,12.176372,12.226372,t,/Volumes/Transcend/USC-TIMIT/EMA/Data/F5/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/F5/wav/u...,f5
56878,3.118639,3.228639,i,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M1/wav/u...,m1
56879,1.474450,1.524450,t,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3
56880,2.857150,2.897150,t,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/mat/u...,/Volumes/Transcend/USC-TIMIT/EMA/Data/M3/wav/u...,m3


In [6]:
for subject in token_df["subject"].unique():
    os.makedirs(f"aeema_scalers", exist_ok=True)
    if os.path.exists(f"aeema_scalers/{subject}.pkl"):
        continue

    subject_token_df = token_df[token_df["subject"]==subject]

    data = {}

    for start, end, art_f in tqdm.tqdm(zip(subject_token_df["start"], subject_token_df["end"], subject_token_df["art_f"]), total=len(subject_token_df)):
        features = get_aeema_articulatory_features(art_f, start, end)
        if features is None:
            continue
        for idx in range(features.shape[1]):
            if idx not in data:
                data[idx] = []
            data[idx].append(features[:, idx])

    scalers = {}
    for idx in tqdm.tqdm(data, total=len(data)):
        data[idx] = np.concatenate(data[idx]).reshape(-1, 1)
        scaler = StandardScaler()
        scaler.fit(data[idx])
        scalers[idx] = scaler

    with open(f"aeema_scalers/{subject}.pkl", "wb") as f:
        pickle.dump(scalers, f)

In [7]:
scalers = {}
for subject in token_df["subject"].unique():
    with open(f"aeema_scalers/{subject}.pkl", "rb") as f:
        scalers[subject] = pickle.load(f)

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from utils.models import SequenceAutoencoder

In [9]:
def temporal_diff(x):
    """
    x: (B, T, ...)
    returns: (B, T-1, ...)
    """
    return x[:, 1:] - x[:, :-1]

def combined_loss(x_hat, x, alpha=0.5):
    """
    alpha: weight on static loss (delta weight = 1-alpha)
    """
    # static loss
    loss_static = F.mse_loss(x_hat, x)

    # delta loss
    dx_hat = temporal_diff(x_hat)
    dx = temporal_diff(x)

    loss_delta = F.mse_loss(dx_hat, dx)

    return alpha * loss_static + (1 - alpha) * loss_delta, loss_static, loss_delta

def latent_stats(z):
    # z: (B, latent_dim)
    return {
        "latent_var_mean": z.var(dim=0).mean().item(),
        "latent_var_min":  z.var(dim=0).min().item(),
        "latent_var_max":  z.var(dim=0).max().item(),
    }

def batch_diversity(x):
    # x: (B, ...)
    x_flat = x.reshape(x.size(0), -1)
    return x_flat.std(dim=0).mean().item()

In [10]:
device = "mps" if torch.backends.mps.is_available() else "cpu"

BATCH_SIZE = 4

In [11]:
log = {
    "batch_idx": [],
    "loss": [],
}
art_model = SequenceAutoencoder(
    feature_shape=(6, 3),
    latent_dim=1024,
).to(device)

art_criterion = nn.MSELoss()
art_optimizer = torch.optim.Adam(art_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

art_model.train()

batch = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    articulatory_features = get_aeema_articulatory_features(art_f, start, end, scalers=scalers[subject])

    if articulatory_features is None:
        continue

    articulatory_features = articulatory_features.reshape(articulatory_features.shape[0], 6, 3)

    batch.append(
        torch.tensor(articulatory_features, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1
    art_batch = torch.stack(batch).to(device)  # (B,T,P,C)
    batch.clear()

    # ===== Articulatory =====
    art_out, art_z = art_model(art_batch)
    if not torch.isfinite(art_out).all():
        raise RuntimeError(
            f"Nonfinite output at batch {batch_idx}"
        )
    art_loss, art_static, art_delta = combined_loss(
        art_out, art_batch
    )

    art_optimizer.zero_grad()
    art_loss.backward()
    art_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(art_loss.item())
    pbar.set_description(
        f"{batch_idx}: ART {art_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if (batch_idx) % 300 == 0:
        with torch.no_grad():
            art_stats = latent_stats(art_z)

        print(
            f"[ART] latent var mean={art_stats['latent_var_mean']:.4e}, "
            f"min={art_stats['latent_var_min']:.4e}"
        )

        print(
            f"[ART] input std={batch_diversity(art_batch):.4e}, "
            f"recon std={batch_diversity(art_out):.4e}"
        )

        torch.save(art_model.state_dict(), "ckpts/aeema/art_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aeema/art_model_training_log.csv", index=False)

torch.save(art_model.state_dict(), "ckpts/aeema/art_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aeema/art_model_training_log.csv", index=False)

300: ART 0.3630:   2%|▏         | 1200/56882 [00:26<19:41, 47.12it/s]

[ART] latent var mean=2.5965e+02, min=6.2033e-03
[ART] input std=8.7309e-01, recon std=4.5494e-01


601: ART 0.5157:   4%|▍         | 2404/56882 [00:54<26:47, 33.90it/s]

[ART] latent var mean=1.0155e+03, min=1.1943e-02
[ART] input std=1.5059e+00, recon std=4.8523e-01


901: ART 0.4322:   6%|▋         | 3606/56882 [01:22<26:26, 33.58it/s]

...

13800: ART 0.3048:  97%|█████████▋| 55264/56882 [20:43<00:46, 34.52it/s]

[ART] latent var mean=8.0350e+04, min=2.8175e-02
[ART] input std=8.4421e-01, recon std=5.3560e-01


14100: ART 0.2995:  99%|█████████▉| 56462/56882 [21:09<00:09, 45.08it/s]

[ART] latent var mean=5.2918e+04, min=7.9283e-01
[ART] input std=6.5915e-01, recon std=5.5147e-01


14204: ART 0.2196: 100%|██████████| 56882/56882 [21:19<00:00, 44.46it/s]


In [12]:
BATCH_SIZE = 4

In [13]:
log = {
    "batch_idx": [],
    "loss": [],
}
ac_model = SequenceAutoencoder(
    feature_shape=30,
    latent_dim=1024,
).to(device)

ac_criterion = nn.MSELoss()
ac_optimizer = torch.optim.Adam(ac_model.parameters(), lr=0.001)

shuffled_token_df = token_df.sample(frac=1, random_state=42).reset_index(drop=True)

pbar = tqdm.tqdm(shuffled_token_df.iterrows(), total=len(shuffled_token_df))

ac_model.train()

batch  = []
batch_idx = 0
for idx, (_, data) in enumerate(pbar):

    start, end, phone, art_f, ac_f, subject = data["start"], data["end"], data["phone"], data["art_f"], data["ac_f"], data["subject"]
    
    mfccs = get_mfccs(ac_f, start, end).T

    if mfccs is None:
        continue

    batch.append(
        torch.tensor(mfccs, dtype=torch.float32)
    )

    # --------------------------------------------------
    # When batch is ready
    # --------------------------------------------------
    if len(batch) < BATCH_SIZE:
        continue
    batch_idx += 1

    ac_batch = torch.stack(batch).to(device)  # (B,T,F)
    batch.clear()

    # ===== Acoustic =====
    ac_out, ac_z = ac_model(ac_batch)
    ac_loss, ac_static, ac_delta = combined_loss(
        ac_out, ac_batch
    )

    ac_optimizer.zero_grad()
    ac_loss.backward()
    ac_optimizer.step()

    log["batch_idx"].append(batch_idx)
    log["loss"].append(ac_loss.item())
    pbar.set_description(
        f"{batch_idx}: AC {ac_loss.item():.4f}"
    )

    # --------------------------------------------------
    # Diagnostics every 1000 updates
    # --------------------------------------------------
    if batch_idx % 1000 == 999:
        with torch.no_grad():
            ac_stats  = latent_stats(ac_z)

        print(
            f"[AC ] latent var mean={ac_stats['latent_var_mean']:.4e}, "
            f"min={ac_stats['latent_var_min']:.4e}"
        )
        print(
            f"[AC ] input std={batch_diversity(ac_batch):.4e}, "
            f"recon std={batch_diversity(ac_out):.4e}"
        )

        torch.save(ac_model.state_dict(), "ckpts/aeema/ac_model_ckpt.pth")
        pd.DataFrame(log).to_csv("ckpts/aeema/ac_model_training_log.csv", index=False)

torch.save(ac_model.state_dict(), "ckpts/aeema/ac_model_ckpt.pth")
pd.DataFrame(log).to_csv("ckpts/aeema/ac_model_training_log.csv", index=False)

1001: AC 104.6397:   7%|▋         | 4002/56882 [00:25<05:50, 150.96it/s]

[AC ] latent var mean=1.1054e+03, min=7.1117e-05
[AC ] input std=1.3015e+01, recon std=6.1465e+00


2002: AC 228.9177:  14%|█▍        | 8007/56882 [00:47<05:53, 138.29it/s]

[AC ] latent var mean=3.0338e+02, min=4.7720e-04
[AC ] input std=1.2769e+01, recon std=3.8959e+00


3003: AC 131.2146:  21%|██        | 11996/56882 [01:10<05:28, 136.47it/s]

...

12999: AC 103.7200:  91%|█████████▏| 51992/56882 [05:19<00:42, 113.93it/s]

[AC ] latent var mean=7.5731e+04, min=6.8245e-04
[AC ] input std=1.1755e+01, recon std=4.9351e+00


14000: AC 109.7840:  98%|█████████▊| 56000/56882 [05:54<00:10, 83.02it/s] 

[AC ] latent var mean=2.0343e+05, min=1.1030e-03
[AC ] input std=1.2374e+01, recon std=6.2211e+00


14220: AC 118.0460: 100%|██████████| 56882/56882 [06:00<00:00, 157.63it/s]
